In [0]:
from pyspark.sql import Row

updates = [
    Row(
        event_id="TEST_001",
        customer_id=999999,
        cell_tower_id="TOWER_999",
        region="Dallas",
        network_type="5G",
        device_type="iPhone",
        event_type="DATA",
        event_timestamp="2026-06-12 10:00:00",
        signal_strength=-55,
        latency_ms=45,
        dropped_call=0,
        data_usage_mb=100.5
    )
]

updates_df = spark.createDataFrame(updates)

display(updates_df)

In [0]:
existing_event = spark.table("workspace.telecom_bronze.network_events").limit(1)
display(existing_event)
from pyspark.sql import Row

updates = [
    Row("6d6a19f9-c17f-4e6c-98af-b9401926e02d", 999999, "TOWER_999", "Dallas", "5G", "iPhone", "DATA", "2026-06-12 10:00:00", -50, 25, 0, 999.99),
    Row("TEST_NEW_001", 888888, "TOWER_888", "Austin", "5G", "Samsung", "CALL", "2026-06-12 11:00:00", -70, 85, 0, 55.75)
]

columns = [
    "event_id", "customer_id", "cell_tower_id", "region", "network_type",
    "device_type", "event_type", "event_timestamp", "signal_strength",
    "latency_ms", "dropped_call", "data_usage_mb"
]

updates_df = spark.createDataFrame(updates, columns)

updates_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.telecom_bronze.network_events_updates"
)

event_id,customer_id,cell_tower_id,region,network_type,device_type,event_type,event_timestamp,signal_strength,latency_ms,dropped_call,data_usage_mb
6d6a19f9-c17f-4e6c-98af-b9401926e02d,131879,TOWER_98,New York,LTE,Motorola,SMS,2026-06-06T08:09:43.000Z,-62,151,0,166.97


In [0]:
spark.sql("""
MERGE INTO workspace.telecom_bronze.network_events t
USING workspace.telecom_bronze.network_events_updates s
ON t.event_id = s.event_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql("""
SELECT COUNT(*) AS total_count
FROM workspace.telecom_bronze.network_events
"""))

total_count
50001


In [0]:
display(spark.sql("""
DESCRIBE HISTORY workspace.telecom_bronze.network_events
"""))


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-06-12T20:53:32.000Z,78451691128077,karth.k1@gmail.com,MERGE,"Map(predicate -> [""(event_id#12268 = event_id#12280)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3141863579845240),756335cd-bf22-4246-b1eb-928d09696208,0612-204655-vnyhb2gu-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3340, numTargetBytesRemoved -> 3340, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3451, materializeSourceTimeMs -> 2, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1715, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1691)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-06-12T20:52:33.000Z,78451691128077,karth.k1@gmail.com,MERGE,"Map(predicate -> [""(event_id#11634 = event_id#11658)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3141863579845240),b757e23c-062e-4dfc-8b5e-4f18c4c7301a,0612-204655-vnyhb2gu-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3340, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 7667, materializeSourceTimeMs -> 9, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 4210, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3312)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-11T20:14:09.000Z,78451691128077,karth.k1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3267654433515104),512fb032-731d-42b5-9b40-296c7aa08d56,0611-200220-uvnecm7g-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50000, numOutputBytes -> 1692692)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
